# 03 — Modeling

## Approach

One anomaly model trained per user — 88 models total.  
Each model is trained exclusively on that user's 10 legitimate sessions.  
Impostor sessions are never seen during training.

Three models compared:
- **Isolation Forest** — primary
- **One-Class SVM** — secondary  
- **Local Outlier Factor** — tertiary

## Evaluation
Ground truth labels held out and used only for evaluation.  
Primary metric: **Equal Error Rate (EER)** — the threshold where False Rejection Rate equals False Acceptance Rate.  
Lower EER = better behavioral fingerprint separation.

Per-user EER computed for all 3 models then averaged across all 88 users.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_curve
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load scaled feature matrix
scaled_df = pd.read_csv('../data/features_kmt_scaled.csv')

feature_cols = [c for c in scaled_df.columns if c not in ('user_id', 'session_id', 'label')]

print(f"Feature matrix shape: {scaled_df.shape}")
print(f"Number of features:   {len(feature_cols)}")
print(f"Users:                {scaled_df['user_id'].nunique()}")
print(f"Sessions per user:    {scaled_df.groupby('user_id').size().unique()}")

Feature matrix shape: (1760, 28)
Number of features:   25
Users:                88
Sessions per user:    [20]


In [24]:
# Core functions
def compute_eer(y_true, anomaly_scores):
    """
    Compute Equal Error Rate (EER).
    
    y_true: 1 = legitimate, 0 = impostor
    anomaly_scores: higher = more anomalous = more likely impostor
    
    Negate scores because roc_curve treats high values as positive 
    class (legitimate=1), but high anomaly score = impostor.
    """
    fpr, tpr, thresholds = roc_curve(y_true, -anomaly_scores)
    frr = 1 - tpr
    far = fpr
    eer_idx = np.argmin(np.abs(frr - far))
    eer = (frr[eer_idx] + far[eer_idx]) / 2
    return eer, frr, far, thresholds


def evaluate_user(user_data, feature_cols, contamination=0.05):
    """
    Train and evaluate all 3 models for a single user.
    Trained on legitimate sessions only.
    Returns EER for each model.
    """
    legitimate = user_data[user_data['label'] == 'legitimate'][feature_cols].values
    impostor = user_data[user_data['label'] == 'impostor'][feature_cols].values

    all_features = np.vstack([legitimate, impostor])
    all_labels = np.array([1] * len(legitimate) + [0] * len(impostor))

    results = {}

    # --- Isolation Forest ---
    iso = IsolationForest(contamination=contamination, random_state=42, n_estimators=100)
    iso.fit(legitimate)
    iso_scores = -iso.decision_function(all_features)
    eer, _, _, _ = compute_eer(all_labels, iso_scores)
    results['isolation_forest'] = eer

    # --- One-Class SVM ---
    ocsvm = OneClassSVM(nu=contamination, kernel='rbf', gamma='scale')
    ocsvm.fit(legitimate)
    ocsvm_scores = -ocsvm.decision_function(all_features)
    eer, _, _, _ = compute_eer(all_labels, ocsvm_scores)
    results['one_class_svm'] = eer

    # --- Local Outlier Factor ---
    lof = LocalOutlierFactor(n_neighbors=5, novelty=True, contamination=contamination)
    lof.fit(legitimate)
    lof_scores = -lof.decision_function(all_features)
    eer, _, _, _ = compute_eer(all_labels, lof_scores)
    results['lof'] = eer

    return results

In [25]:
# Feature selection
# Select top features by mean absolute difference across all users
raw_df = pd.read_csv('../data/features_kmt.csv')
feature_cols = [c for c in raw_df.columns if c not in ('user_id', 'session_id', 'label')]

diff_records = []
for user_id, user_data in raw_df.groupby('user_id'):
    leg = user_data[user_data['label'] == 'legitimate'][feature_cols]
    imp = user_data[user_data['label'] == 'impostor'][feature_cols]
    diff = (imp.mean() - leg.mean()).abs()
    diff_records.append(diff)

mean_diffs = pd.DataFrame(diff_records).mean().sort_values(ascending=False)

# Keystroke-only features — mouse velocity inverts signal due to scale dominance
keystroke_features = [
    'dwell_median',
    'dwell_mean',
    'dwell_std',
    'dwell_max',
    'flight_median',
    'flight_min',
    'click_dwell_mean'
]

print("Selected features:")
for f in keystroke_features:
    print(f"  {f:<25} mean diff: {mean_diffs[f]:.4f}")

Selected features:
  dwell_median              mean diff: 0.0194
  dwell_mean                mean diff: 0.0183
  dwell_std                 mean diff: 0.0182
  dwell_max                 mean diff: 0.1071
  flight_median             mean diff: 0.1066
  flight_min                mean diff: 0.0312
  click_dwell_mean          mean diff: 0.0437


In [26]:
# Single user-sanity check
# Sanity check on user 1 before full evaluation
user_1_raw = raw_df[raw_df['user_id'] == 1]
test_results = evaluate_user(user_1_raw, keystroke_features)

print("EER for user 1:")
for model, eer in test_results.items():
    print(f"  {model:<20}: {eer:.4f} ({eer*100:.2f}%)")
print("\nNote: Lower EER = better. Random chance = 0.50 (50%)")

EER for user 1:
  isolation_forest    : 0.1000 (10.00%)
  one_class_svm       : 0.0500 (5.00%)
  lof                 : 0.3000 (30.00%)

Note: Lower EER = better. Random chance = 0.50 (50%)


In [27]:
# full evaluation across all 88 users
# Run all 3 models across all 88 users
all_results = []

for user_id, user_data in tqdm(raw_df.groupby('user_id'), desc="Evaluating users"):
    results = evaluate_user(user_data, keystroke_features)
    results['user_id'] = user_id
    all_results.append(results)

results_df = pd.DataFrame(all_results)

print("=== EER SUMMARY ACROSS ALL 88 USERS ===\n")
for model in ['isolation_forest', 'one_class_svm', 'lof']:
    eers = results_df[model]
    print(f"{model}:")
    print(f"  Mean EER:        {eers.mean():.4f} ({eers.mean()*100:.2f}%)")
    print(f"  Median EER:      {eers.median():.4f} ({eers.median()*100:.2f}%)")
    print(f"  Best EER:        {eers.min():.4f} ({eers.min()*100:.2f}%)")
    print(f"  Worst EER:       {eers.max():.4f} ({eers.max()*100:.2f}%)")
    print(f"  Users < 50% EER: {(eers < 0.5).sum()}/88\n")

Evaluating users: 100%|██████████| 88/88 [00:16<00:00,  5.31it/s]

=== EER SUMMARY ACROSS ALL 88 USERS ===

isolation_forest:
  Mean EER:        0.1727 (17.27%)
  Median EER:      0.1500 (15.00%)
  Best EER:        0.0000 (0.00%)
  Worst EER:       0.5500 (55.00%)
  Users < 50% EER: 87/88

one_class_svm:
  Mean EER:        0.1085 (10.85%)
  Median EER:      0.0500 (5.00%)
  Best EER:        0.0000 (0.00%)
  Worst EER:       0.6000 (60.00%)
  Users < 50% EER: 86/88

lof:
  Mean EER:        0.1574 (15.74%)
  Median EER:      0.1500 (15.00%)
  Best EER:        0.0000 (0.00%)
  Worst EER:       0.4500 (45.00%)
  Users < 50% EER: 88/88



In [ ]:
# Visualie EER Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['isolation_forest', 'one_class_svm', 'lof']
titles = ['Isolation Forest', 'One-Class SVM', 'Local Outlier Factor']
colors = ['steelblue', 'tomato', 'seagreen']

for ax, model, title, color in zip(axes, models, titles, colors):
    eers = results_df[model] * 100
    ax.hist(eers, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(eers.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {eers.mean():.1f}%')
    ax.axvline(50, color='red', linestyle=':', linewidth=1, label='Random (50%)')
    ax.set_title(title)
    ax.set_xlabel('EER (%)')
    ax.set_ylabel('Number of Users')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 70)

plt.suptitle('EER Distribution Across 88 Users — All Models', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/04_eer_distributions.png', dpi=150)
plt.show()